<a href="https://colab.research.google.com/github/arthursrosa-star/PDM_ML_SPARK_PRK/blob/SparkMl/PDM_Classificacao_Spark.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Classificação de acidentes fatais em rodovias federais com Apache Spark ML

**Disciplina:** Processamento de Dados Massivos  
**Tecnologia:** PySpark + Spark MLlib  
**Fonte:** Dados Abertos da Polícia Rodoviária Federal (PRF)  
**Período proposto:** 2022–2025  
**Problema:** classificação binária — identificar se um acidente registrado foi fatal.

> Execute este notebook no Google Colab, de cima para baixo. A primeira execução pode demorar porque os arquivos anuais são baixados e descompactados.

## 1. Contextualização e pergunta de negócio

A base da PRF reúne registros de acidentes ocorridos em rodovias federais brasileiras. Neste trabalho, cada linha representa uma **ocorrência de acidente**. O objetivo é construir um classificador que estime se a ocorrência foi fatal (`1`) ou não fatal (`0`), usando características do local, da via, do momento e da dinâmica do acidente.

**Pergunta de negócio:** considerando as características observáveis de uma ocorrência, é possível classificar se ela pertence à classe de acidentes fatais?

**Por que é um problema de dados massivos?** A PRF disponibiliza séries históricas nacionais, atualizadas periodicamente, com registros de diferentes unidades da federação e diversas variáveis. A consolidação de vários anos aumenta o volume de dados e exige leitura, limpeza, agregações e transformações distribuídas. O Spark permite executar essas operações de forma paralela e escalável, sem depender do processamento linha a linha em memória com pandas.

**Unidade de análise:** acidente/ocorrência (não pessoa envolvida).  
**Amostragem:** o notebook combina os arquivos anuais de 2022, 2023, 2024 e 2025 disponíveis no portal da PRF. Não é uma amostra aleatória: é um recorte temporal de quatro anos. O número efetivo de registros será calculado após a ingestão.

**Atenção metodológica:** a classificação fatal é derivada do campo `classificacao_acidente`. Campos que revelam diretamente o resultado do acidente — como `mortos`, `feridos`, `feridos_graves`, `feridos_leves`, `ilesos`, `ignorados` e a própria `classificacao_acidente` — não serão usados como preditores, para evitar vazamento de informação (*data leakage*).

### Fonte oficial e arquivos

Portal oficial: [Dados Abertos da PRF – BAT: Boletim de Acidente de Trânsito](https://www.gov.br/prf/pt-br/acesso-a-informacao/dados-abertos/dados-abertos-da-prf).

Os links abaixo correspondem aos arquivos de acidentes **agrupados por ocorrência** disponibilizados pela própria página da PRF. O notebook baixa os ZIPs diretamente do Google Drive indicado pelo portal.

In [ ]:
# 2. Preparação do ambiente no Google Colab
!pip -q install pyspark==3.5.1 gdown

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dataproc-spark-connect 1.1.0 requires pyspark[connect]~=4.0.0, but you have pyspark 3.5.1 which is incompatible.


In [ ]:
import os
import re
import glob
import zipfile
import shutil
import gdown
from pathlib import Path

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler, Imputer, StandardScaler
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator, BinaryClassificationEvaluator
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator

# Configuração adequada ao ambiente gratuito do Colab.
spark = (
    SparkSession.builder
    .appName("PRF_Acidentes_Classificacao")
    .config("spark.driver.memory", "4g")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print("Versão do Spark:", spark.version)

Versão do Spark: 3.5.1


## 3. Ingestão dos dados com PySpark

Os arquivos oficiais são disponibilizados compactados. Para manter a leitura reproduzível, o notebook baixa os quatro anos a partir dos identificadores publicados nos links da página da PRF, descompacta-os e faz a leitura com Spark.

O `inferSchema=True` é utilizado porque os arquivos anuais são publicados como CSV e podem conter pequenas diferenças de tipos/representação entre anos. Após a leitura, as colunas numéricas e a data são convertidas explicitamente. A separação por `;` e a codificação `ISO-8859-1` são especificadas para a leitura dos CSVs da PRF.

In [ ]:
# Identificadores dos arquivos ZIP ligados na página oficial da PRF.
ARQUIVOS_PRF = {
    2022: "1PRQjuV5gOn_nn6UNvaJyVURDIfbSAK4-",
    2023: "1-WO3SfNrwwZ5_l7fRTiwBKRw7mi1-HUq",
    2024: "14lB0vqMFkaZj8HZ44b0njYgxs9nAN8KO",
    2025: "1-G3MdmHBt6CprDwcW99xxC4BZ2DU5ryR",
}

PASTA_DADOS = Path("/content/dados_prf")
PASTA_DADOS.mkdir(parents=True, exist_ok=True)

for ano, file_id in ARQUIVOS_PRF.items():
    pasta_ano = PASTA_DADOS / str(ano)
    pasta_ano.mkdir(exist_ok=True)

    # Evita baixar novamente se o CSV já tiver sido extraído.
    csv_existente = list(pasta_ano.rglob("*.csv"))
    if csv_existente:
        print(f"{ano}: arquivo já extraído.")
        continue

    zip_path = PASTA_DADOS / f"datatran{ano}.zip"
    if not zip_path.exists():
        url = f"https://drive.google.com/uc?id={file_id}"
        print(f"Baixando {ano}...")
        gdown.download(url, str(zip_path), quiet=False)

    if not zip_path.exists() or zip_path.stat().st_size == 0:
        raise RuntimeError(f"Falha no download do arquivo de {ano}.")

    print(f"Extraindo {ano}...")
    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall(pasta_ano)

arquivos_csv = sorted(glob.glob(str(PASTA_DADOS / "**" / "*.csv"), recursive=True))
print("Quantidade de arquivos CSV encontrados:", len(arquivos_csv))
for arquivo in arquivos_csv:
    print(arquivo)

Baixando 2022...


Downloading...
From: https://drive.google.com/uc?id=1PRQjuV5gOn_nn6UNvaJyVURDIfbSAK4-
To: /content/dados_prf/datatran2022.zip
100%|██████████| 3.23M/3.23M [00:00<00:00, 48.9MB/s]


Extraindo 2022...
Baixando 2023...


Downloading...
From: https://drive.google.com/uc?id=1-WO3SfNrwwZ5_l7fRTiwBKRw7mi1-HUq
To: /content/dados_prf/datatran2023.zip
100%|██████████| 3.40M/3.40M [00:00<00:00, 43.6MB/s]


Extraindo 2023...
Baixando 2024...


Downloading...
From: https://drive.google.com/uc?id=14lB0vqMFkaZj8HZ44b0njYgxs9nAN8KO
To: /content/dados_prf/datatran2024.zip
100%|██████████| 3.78M/3.78M [00:00<00:00, 26.1MB/s]


Extraindo 2024...
Baixando 2025...


Downloading...
From: https://drive.google.com/uc?id=1-G3MdmHBt6CprDwcW99xxC4BZ2DU5ryR
To: /content/dados_prf/datatran2025.zip
100%|██████████| 3.73M/3.73M [00:00<00:00, 51.0MB/s]

Extraindo 2025...
Quantidade de arquivos CSV encontrados: 4
/content/dados_prf/2022/datatran2022.csv
/content/dados_prf/2023/datatran2023.csv
/content/dados_prf/2024/datatran2024.csv
/content/dados_prf/2025/datatran2025.csv


In [ ]:
# Leitura distribuída dos arquivos anuais.
# header=True: primeira linha contém os nomes das colunas.
# inferSchema=True: inferência inicial; tipos críticos serão tratados explicitamente adiante.
df_bruto = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .option("sep", ";")
    .option("encoding", "ISO-8859-1")
    .option("quote", '"')
    .option("escape", '"')
    .csv(arquivos_csv)
)

print("Registros:", df_bruto.count())
print("Colunas:", len(df_bruto.columns))
df_bruto.printSchema()
df_bruto.show(5, truncate=False)

Registros: 278103
Colunas: 30
root
 |-- id: double (nullable = true)
 |-- data_inversa: date (nullable = true)
 |-- dia_semana: string (nullable = true)
 |-- horario: timestamp (nullable = true)
 |-- uf: string (nullable = true)
 |-- br: integer (nullable = true)
 |-- km: string (nullable = true)
 |-- municipio: string (nullable = true)
 |-- causa_acidente: string (nullable = true)
 |-- tipo_acidente: string (nullable = true)
 |-- classificacao_acidente: string (nullable = true)
 |-- fase_dia: string (nullable = true)
 |-- sentido_via: string (nullable = true)
 |-- condicao_metereologica: string (nullable = true)
 |-- tipo_pista: string (nullable = true)
 |-- tracado_via: string (nullable = true)
 |-- uso_solo: string (nullable = true)
 |-- pessoas: integer (nullable = true)
 |-- mortos: integer (nullable = true)
 |-- feridos_leves: integer (nullable = true)
 |-- feridos_graves: integer (nullable = true)
 |-- ilesos: integer (nullable = true)
 |-- ignorados: integer (nullable = true)
 

## 4. Exploração inicial

Nesta etapa são examinados o tamanho da base, os nomes das colunas, os tipos inferidos, estatísticas descritivas e a distribuição inicial da variável de classificação. As operações são executadas no Spark.

In [ ]:
# Padroniza nomes das colunas para facilitar o uso no pipeline.
def normalizar_nome(nome):
    nome = nome.strip().lower()
    nome = (nome.replace(" ", "_")
                .replace("-", "_")
                .replace("/", "_"))
    return nome

df = df_bruto
for nome in df.columns:
    novo = normalizar_nome(nome)
    if nome != novo:
        df = df.withColumnRenamed(nome, novo)

print("Dimensões:", df.count(), "linhas x", len(df.columns), "colunas")
print("Colunas disponíveis:")
print(df.columns)

Dimensões: 278103 linhas x 30 colunas
Colunas disponíveis:
['id', 'data_inversa', 'dia_semana', 'horario', 'uf', 'br', 'km', 'municipio', 'causa_acidente', 'tipo_acidente', 'classificacao_acidente', 'fase_dia', 'sentido_via', 'condicao_metereologica', 'tipo_pista', 'tracado_via', 'uso_solo', 'pessoas', 'mortos', 'feridos_leves', 'feridos_graves', 'ilesos', 'ignorados', 'feridos', 'veiculos', 'latitude', 'longitude', 'regional', 'delegacia', 'uop']


In [ ]:
# Estatísticas descritivas das colunas numéricas que existirem na base.
colunas_numericas = [
    c for c, t in df.dtypes
    if t in ("int", "bigint", "double", "float", "smallint", "tinyint")
]
if colunas_numericas:
    df.select(*colunas_numericas).describe().show(truncate=False)

# Distribuição da classificação original.
if "classificacao_acidente" in df.columns:
    (df.groupBy("classificacao_acidente")
       .count()
       .orderBy(F.desc("count"))
       .show(truncate=False))
else:
    raise ValueError("A coluna 'classificacao_acidente' não foi encontrada. Confira o schema do CSV.")

+-------+-----------------+------------------+------------------+-------------------+------------------+-------------------+------------------+------------------+------------------+------------------+
|summary|id               |br                |pessoas           |mortos             |feridos_leves     |feridos_graves     |ilesos            |ignorados         |feridos           |veiculos          |
+-------+-----------------+------------------+------------------+-------------------+------------------+-------------------+------------------+------------------+------------------+------------------+
|count  |278103           |278103            |278103            |278103             |278103            |278103             |278103            |278103            |278103            |278103            |
|mean   |580888.104935941 |209.80631996059014|2.59251428427597  |0.08368841760067314|0.8707277519480192|0.27874924038935217|1.0479606476737036|0.3986832216840523|1.1494769923373713|1.9899174047025

## 5. Limpeza e preparação da variável-alvo

A variável-alvo será binária:

- `1`: acidente fatal — `classificacao_acidente` igual a “Com Vítimas Fatais”;
- `0`: demais acidentes registrados.

Registros sem classificação serão removidos, pois não permitem determinar o rótulo. Duplicatas exatas serão removidas. Valores inválidos em variáveis numéricas serão convertidos para nulos e tratados posteriormente pelo `Imputer`.

O notebook também verifica a proporção das classes. Em problemas de acidentes fatais, é esperado que a classe positiva seja minoritária; por isso, a avaliação não ficará restrita à acurácia.

In [ ]:
# Conversão robusta de campos numéricos, se estiverem presentes.
campos_numericos = ["km", "pessoas", "mortos", "feridos_leves",
                    "feridos_graves", "ilesos", "ignorados",
                    "feridos", "veiculos"]

for c in campos_numericos:
    if c in df.columns:
        df = df.withColumn(
            c,
            F.regexp_replace(F.col(c).cast("string"), ",", ".").cast("double")
        )

# Remove duplicatas exatas com operação distribuída.
linhas_antes = df.count()
df = df.dropDuplicates()
linhas_depois = df.count()
print("Duplicatas removidas:", linhas_antes - linhas_depois)

# Padronização textual da classe.
df = df.withColumn(
    "classificacao_norm",
    F.lower(F.trim(F.col("classificacao_acidente").cast("string")))
)

# Mantém somente categorias reconhecíveis para criar o alvo.
df = df.filter(F.col("classificacao_norm").isNotNull())

df = df.withColumn(
    "label",
    F.when(F.col("classificacao_norm").contains("fatais"), F.lit(1.0))
     .otherwise(F.lit(0.0))
)

print("Distribuição do alvo:")
(df.groupBy("label").count()
   .withColumn("percentual", F.round(100 * F.col("count") / F.sum("count").over(
       __import__("pyspark").sql.Window.partitionBy()
   ), 2))
   .orderBy("label")
   .show())

Duplicatas removidas: 0
Distribuição do alvo:
+-----+------+----------+
|label| count|percentual|
+-----+------+----------+
|  0.0|258147|     92.82|
|  1.0| 19956|      7.18|
+-----+------+----------+



## 6. Engenharia de atributos

Serão utilizadas variáveis disponíveis no registro que descrevem o contexto do acidente:

- **Categóricas:** UF, dia da semana, horário/fase do dia, BR, município, causa, tipo de acidente, condição meteorológica, tipo de pista, traçado, sentido da via e uso do solo;
- **Numéricas:** quilômetro da rodovia e, quando disponíveis, outras medidas de contexto que não representem o resultado do acidente.

A data será convertida em atributos de calendário (ano, mês e dia da semana). Campos de vítimas e a classificação original são excluídos dos preditores por constituírem vazamento do alvo.

A lista de variáveis é montada de forma defensiva: se algum campo não existir em determinada versão do arquivo, ele será ignorado.

In [ ]:
# Data: cria ano e mês a partir de data_inversa, quando disponível.
if "data_inversa" in df.columns:
    df = df.withColumn(
        "data_ref",
        F.coalesce(
            F.to_date("data_inversa", "yyyy-MM-dd"),
            F.to_date("data_inversa", "dd/MM/yyyy"),
            F.to_date("data_inversa", "MM/dd/yyyy")
        )
    )
    df = (df.withColumn("ano", F.year("data_ref").cast("double"))
            .withColumn("mes", F.month("data_ref").cast("double")))

# Hora: horario pode vir como timestamp completo (ex.: 2026-11-04 15:45:00).
# F.hour() extrai corretamente o componente de hora, sem confundir com o ano.
if "horario" in df.columns:
    df = df.withColumn(
        "hora",
        F.hour(F.to_timestamp(F.col("horario"))).cast("double")
    )
    df = df.withColumn(
        "hora",
        F.when(F.col("hora").between(0, 23), F.col("hora")).otherwise(None)
    )

    print("Distribuição resumida da hora extraída:")
    df.select("horario", "hora").show(5, truncate=False)
    df.select(
        F.count("hora").alias("horarios_validos"),
        F.min("hora").alias("hora_minima"),
        F.max("hora").alias("hora_maxima")
    ).show()

# Campos candidatos por tipo.
candidatas_cat = [
    "uf", "dia_semana", "br", "municipio", "causa_acidente",
    "tipo_acidente", "fase_dia", "sentido_via",
    "condicao_metereologica", "condicao_meteorologica",
    "tipo_pista", "tracado_via", "uso_solo"
]
candidatas_num = ["km", "ano", "mes", "hora"]

cat_cols = [c for c in candidatas_cat if c in df.columns]
num_cols = [c for c in candidatas_num if c in df.columns]

print("Categóricas selecionadas:", cat_cols)
print("Numéricas selecionadas:", num_cols)

# Não usar identificadores, classificação nem informações de vítimas.
colunas_proibidas = {
    "id", "data_inversa", "data_ref", "horario",
    "classificacao_acidente", "classificacao_norm",
    "mortos", "feridos", "feridos_leves", "feridos_graves",
    "ilesos", "ignorados", "pessoas", "veiculos",
    "label"
}
cat_cols = [c for c in cat_cols if c not in colunas_proibidas]
num_cols = [c for c in num_cols if c not in colunas_proibidas]

# Mantém apenas registros com pelo menos algum atributo candidato.
if not cat_cols and not num_cols:
    raise ValueError("Nenhuma variável preditora foi identificada. Verifique os nomes das colunas.")

# Remove linhas sem nenhuma informação útil nos preditores selecionados.
condicao_algum_dado = None
for c in cat_cols + num_cols:
    atual = F.col(c).isNotNull()
    condicao_algum_dado = atual if condicao_algum_dado is None else (condicao_algum_dado | atual)

df_modelo = df.filter(condicao_algum_dado).select(
    *cat_cols, *num_cols, "label"
)

# Descarta valores categóricos vazios como nulos.
for c in cat_cols:
    df_modelo = df_modelo.withColumn(
        c,
        F.when(F.trim(F.col(c).cast("string")) == "", None)
         .otherwise(F.col(c).cast("string"))
    )

print("Registros após seleção de atributos:", df_modelo.count())
df_modelo.show(5, truncate=False)

Distribuição resumida da hora extraída:
+-------------------+----+
|horario            |hora|
+-------------------+----+
|2026-11-04 15:45:00|15.0|
|2026-11-04 18:10:00|18.0|
|2026-11-04 17:30:00|17.0|
|2026-11-04 05:00:00|5.0 |
|2026-11-04 06:50:00|6.0 |
+-------------------+----+
only showing top 5 rows

+----------------+-----------+-----------+
|horarios_validos|hora_minima|hora_maxima|
+----------------+-----------+-----------+
|          278103|        0.0|       23.0|
+----------------+-----------+-----------+

Categóricas selecionadas: ['uf', 'dia_semana', 'br', 'municipio', 'causa_acidente', 'tipo_acidente', 'fase_dia', 'sentido_via', 'condicao_metereologica', 'tipo_pista', 'tracado_via', 'uso_solo']
Numéricas selecionadas: ['km', 'ano', 'mes', 'hora']
Registros após seleção de atributos: 278103
+---+-------------+---+--------------------+----------------------------------------+-------------------------+---------+-----------+----------------------+----------+-------------+---

## 7. Separação treino/teste e Pipeline Spark ML

A separação é feita antes do ajuste dos transformadores e modelos. Assim, indexadores, imputadores, codificadores e modelo são ajustados somente no conjunto de treino, evitando vazamento de informação do teste.

O pipeline contém:

1. `StringIndexer` para transformar categorias em índices;
2. `OneHotEncoder` para codificar as categorias;
3. `Imputer` para preencher valores numéricos ausentes pela mediana;
4. `VectorAssembler` para montar o vetor de atributos;
5. `StandardScaler` para padronizar os atributos;
6. classificador Spark ML.

Como a classe fatal pode ser minoritária, será calculado um peso inversamente proporcional à frequência das classes no treino. Esse peso será utilizado pelo classificador.

**Verificação adicional de qualidade:** antes de criar o `Imputer`, o notebook verifica quais variáveis numéricas possuem ao menos um valor válido no conjunto de treino. Colunas inteiramente nulas/NaN no treino são removidas automaticamente do Pipeline, pois não existe mediana que possa ser estimada para elas. Essa verificação ocorre antes do ajuste do Pipeline e usa somente o treino.


In [ ]:
# Divisão treino/teste antes do ajuste dos transformadores.
treino, teste = df_modelo.randomSplit([0.8, 0.2], seed=42)

# Cache para evitar recomputar a leitura e as transformações em múltiplas ações.
treino = treino.cache()
teste = teste.cache()

print("Treino:", treino.count())
print("Teste:", teste.count())

# IMPORTANTE:
# O Imputer não consegue calcular mediana de uma coluna totalmente vazia.
# A verificação é feita APENAS no treino para não usar informação do teste.
num_cols_validas = []
for c in num_cols:
    qtd_validos = treino.filter(
        F.col(c).isNotNull() & (~F.isnan(F.col(c)))
    ).count()

    if qtd_validos > 0:
        num_cols_validas.append(c)
        print(f"{c}: {qtd_validos} valores numéricos válidos no treino.")
    else:
        print(f"{c}: removida do Pipeline (todos os valores são nulos/NaN no treino).")

num_cols = num_cols_validas
print("Variáveis numéricas mantidas:", num_cols)

if not cat_cols and not num_cols:
    raise ValueError("Nenhuma variável preditora válida permaneceu após a verificação.")

# Peso de classe calculado exclusivamente no treino.
contagens = {float(r["label"]): r["count"] for r in treino.groupBy("label").count().collect()}
total_treino = sum(contagens.values())
n_classes = len(contagens)

if n_classes < 2:
    raise ValueError(
        f"O conjunto de treino contém apenas uma classe: {contagens}. "
        "Altere a semente ou a proporção da divisão treino/teste."
    )

pesos = {
    classe: total_treino / (n_classes * quantidade)
    for classe, quantidade in contagens.items()
}
print("Contagens no treino:", contagens)
print("Pesos calculados:", pesos)

def adicionar_peso(dataframe):
    expr = F.lit(1.0)
    for classe, peso in pesos.items():
        expr = F.when(
            F.col("label") == float(classe),
            F.lit(float(peso))
        ).otherwise(expr)
    return dataframe.withColumn("peso_classe", expr)

treino_pesado = adicionar_peso(treino)


Treino: 222651
Teste: 55452
km: 222651 valores numéricos válidos no treino.
ano: 222651 valores numéricos válidos no treino.
mes: 222651 valores numéricos válidos no treino.
hora: 222651 valores numéricos válidos no treino.
Variáveis numéricas mantidas: ['km', 'ano', 'mes', 'hora']
Contagens no treino: {1.0: 15873, 0.0: 206778}
Pesos calculados: {1.0: 7.013513513513513, 0.0: 0.5383817427385892}


In [ ]:
# Etapas de pré-processamento.
indexers = [
    StringIndexer(
        inputCol=c,
        outputCol=f"{c}_idx",
        handleInvalid="keep"
    )
    for c in cat_cols
]

indexed_cols = [f"{c}_idx" for c in cat_cols]

encoder = OneHotEncoder(
    inputCols=indexed_cols,
    outputCols=[f"{c}_ohe" for c in cat_cols],
    handleInvalid="keep"
) if cat_cols else None

ohe_cols = [f"{c}_ohe" for c in cat_cols]

imputador = Imputer(
    inputCols=num_cols,
    outputCols=[f"{c}_imputado" for c in num_cols],
    strategy="median"
) if num_cols else None

num_imputadas = [f"{c}_imputado" for c in num_cols]
atributos_entrada = ohe_cols + num_imputadas

assembler = VectorAssembler(
    inputCols=atributos_entrada,
    outputCol="features_brutas",
    handleInvalid="keep"
)

scaler = StandardScaler(
    inputCol="features_brutas",
    outputCol="features",
    withStd=True,
    withMean=False
)

preprocessamento = indexers.copy()
if encoder is not None:
    preprocessamento.append(encoder)
if imputador is not None:
    preprocessamento.append(imputador)
preprocessamento += [assembler, scaler]

print("Pipeline de pré-processamento criado.")

Pipeline de pré-processamento criado.


## 8. Modelagem

Serão comparados dois algoritmos de classificação disponíveis no Spark ML:

- **Regressão Logística:** modelo linear, adequado como referência interpretável para classificação binária;
- **Random Forest:** conjunto de árvores de decisão, capaz de capturar relações não lineares e interações entre atributos.

A comparação será realizada no mesmo conjunto de teste. A escolha final deve considerar principalmente F1 e AUC, além da matriz de confusão, pois a acurácia isolada pode ser enganosa em bases desbalanceadas.

In [ ]:
# Modelo 1: Regressão Logística.
lr = LogisticRegression(
    featuresCol="features",
    labelCol="label",
    weightCol="peso_classe",
    maxIter=50,
    regParam=0.05,
    elasticNetParam=0.0
)

pipeline_lr = Pipeline(stages=preprocessamento + [lr])
modelo_lr = pipeline_lr.fit(treino_pesado)

# Modelo 2: Random Forest.
rf = RandomForestClassifier(
    featuresCol="features",
    labelCol="label",
    weightCol="peso_classe",
    numTrees=80,
    maxDepth=10,
    seed=42
)

pipeline_rf = Pipeline(stages=preprocessamento + [rf])
modelo_rf = pipeline_rf.fit(treino_pesado)

print("Modelos treinados.")

Modelos treinados.


## 9. Avaliação e interpretação

Métricas utilizadas:

- **AUC-ROC:** capacidade de separar as duas classes em diferentes limiares;
- **F1-score:** equilíbrio entre precisão e revocação da classe positiva;
- **Precisão:** proporção das previsões de fatalidade que realmente foram fatais;
- **Recall:** proporção dos acidentes fatais identificados pelo modelo;
- **Matriz de confusão:** detalha verdadeiros/falsos positivos e negativos.

Para a classe fatal, recall é particularmente relevante: um falso negativo representa um acidente fatal classificado como não fatal. Entretanto, aumentar o recall pode elevar falsos positivos; a decisão depende do uso pretendido do modelo.

In [ ]:
def avaliar_modelo(modelo, dados_teste, nome):
    pred = modelo.transform(dados_teste).cache()

    evaluator_f1 = MulticlassClassificationEvaluator(
        labelCol="label", predictionCol="prediction", metricName="f1"
    )
    evaluator_acc = MulticlassClassificationEvaluator(
        labelCol="label", predictionCol="prediction", metricName="accuracy"
    )
    evaluator_auc = BinaryClassificationEvaluator(
        labelCol="label", rawPredictionCol="rawPrediction",
        metricName="areaUnderROC"
    )

    f1 = evaluator_f1.evaluate(pred)
    acuracia = evaluator_acc.evaluate(pred)
    auc = evaluator_auc.evaluate(pred)

    # Métricas específicas da classe positiva.
    tp = pred.filter((F.col("label") == 1) & (F.col("prediction") == 1)).count()
    tn = pred.filter((F.col("label") == 0) & (F.col("prediction") == 0)).count()
    fp = pred.filter((F.col("label") == 0) & (F.col("prediction") == 1)).count()
    fn = pred.filter((F.col("label") == 1) & (F.col("prediction") == 0)).count()

    precisao = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    f1_positivo = (2 * precisao * recall / (precisao + recall)
                   if (precisao + recall) else 0.0)

    print(f"\n===== {nome} =====")
    print(f"AUC-ROC: {auc:.4f}")
    print(f"F1 ponderado: {f1:.4f}")
    print(f"F1 classe fatal: {f1_positivo:.4f}")
    print(f"Precisão fatal: {precisao:.4f}")
    print(f"Recall fatal: {recall:.4f}")
    print(f"Acurácia: {acuracia:.4f}")
    print("Matriz de confusão (linhas = real; colunas = previsto)")
    print(f"                 Previsto 0   Previsto 1")
    print(f"Real 0 (não fatal)    {tn:8d}     {fp:8d}")
    print(f"Real 1 (fatal)        {fn:8d}     {tp:8d}")

    pred.unpersist()
    return {
        "modelo": nome, "auc": auc, "f1_ponderado": f1,
        "f1_fatal": f1_positivo, "precisao_fatal": precisao,
        "recall_fatal": recall, "acuracia": acuracia,
        "TN": tn, "FP": fp, "FN": fn, "TP": tp
    }

resultado_lr = avaliar_modelo(modelo_lr, teste, "Regressão Logística")
resultado_rf = avaliar_modelo(modelo_rf, teste, "Random Forest")


===== Regressão Logística =====
AUC-ROC: 0.8049
F1 ponderado: 0.8297
F1 classe fatal: 0.3148
Precisão fatal: 0.2049
Recall fatal: 0.6787
Acurácia: 0.7824
Matriz de confusão (linhas = real; colunas = previsto)
                 Previsto 0   Previsto 1
Real 0 (não fatal)       40616        10753
Real 1 (fatal)            1312         2771

===== Random Forest =====
AUC-ROC: 0.7890
F1 ponderado: 0.8082
F1 classe fatal: 0.2889
Precisão fatal: 0.1830
Recall fatal: 0.6858
Acurácia: 0.7515
Matriz de confusão (linhas = real; colunas = previsto)
                 Previsto 0   Previsto 1
Real 0 (não fatal)       38871        12498
Real 1 (fatal)            1283         2800


In [ ]:
# Comparação consolidada.
resultados = spark.createDataFrame([resultado_lr, resultado_rf])
resultados.select(
    "modelo", "auc", "f1_ponderado", "f1_fatal",
    "precisao_fatal", "recall_fatal", "acuracia"
).show(truncate=False)

+-------------------+------------------+------------------+------------------+-------------------+------------------+------------------+
|modelo             |auc               |f1_ponderado      |f1_fatal          |precisao_fatal     |recall_fatal      |acuracia          |
+-------------------+------------------+------------------+------------------+-------------------+------------------+------------------+
|Regressão Logística|0.8049437863260815|0.8297486712445091|0.3147611745328563|0.20489500147885242|0.6786676463384766|0.7824244391545842|
|Random Forest      |0.7889689104863337|0.8081567606898868|0.288942779010371 |0.18303046149823507|0.6857702669605682|0.7514787564019332|
+-------------------+------------------+------------------+------------------+-------------------+------------------+------------------+



## 10. Persistência do modelo

O modelo com maior AUC no conjunto de teste será salvo usando o método `save()` do Spark ML. O diretório será gravado no armazenamento do Colab. Para manter o artefato após o encerramento da sessão, pode-se copiar o diretório para o Google Drive.

In [ ]:
# Seleciona o melhor modelo pela AUC medida no conjunto de teste.
if resultado_rf["auc"] >= resultado_lr["auc"]:
    melhor_modelo = modelo_rf
    nome_melhor = "RandomForest"
else:
    melhor_modelo = modelo_lr
    nome_melhor = "LogisticRegression"

CAMINHO_MODELO = f"/content/modelo_prf_{nome_melhor}"
if os.path.exists(CAMINHO_MODELO):
    shutil.rmtree(CAMINHO_MODELO)

melhor_modelo.write().overwrite().save(CAMINHO_MODELO)
print("Modelo salvo em:", CAMINHO_MODELO)
print("Modelo selecionado:", nome_melhor)

Modelo salvo em: /content/modelo_prf_LogisticRegression
Modelo selecionado: LogisticRegression


## 11. Conclusão

Este trabalho implementou um fluxo de classificação binária de acidentes fatais em dados da PRF com Apache Spark: ingestão distribuída, exploração, limpeza, engenharia de atributos, pipeline de transformação, treinamento de classificadores, avaliação e persistência.

A conclusão final deve ser preenchida com os resultados obtidos no ambiente Colab. Discuta o desempenho em AUC e F1, especialmente para a classe fatal, os erros da matriz de confusão e as limitações dos dados.

**Limitações do estudo:**
- o recorte de 2022–2025 não representa necessariamente todos os períodos e condições das rodovias;
- mudanças de preenchimento e registro podem afetar a comparabilidade temporal;
- campos ausentes e categorias pouco frequentes podem reduzir a qualidade das previsões;
- o modelo não estabelece relações causais;
- a classificação não deve ser usada isoladamente para decisões operacionais ou de segurança pública.

**Possíveis melhorias:** testar validação temporal (treinar em anos anteriores e testar no ano mais recente), ajustar hiperparâmetros, comparar outros classificadores do Spark ML, estudar limiares de decisão e avaliar o desempenho separadamente por UF e tipo de rodovia.

## 12. Referências

- Polícia Rodoviária Federal (PRF). **Dados Abertos da PRF – BAT: Boletim de Acidente de Trânsito**. https://www.gov.br/prf/pt-br/acesso-a-informacao/dados-abertos/dados-abertos-da-prf
- Polícia Rodoviária Federal (PRF). **Dicionário de Dados – Acidentes**. https://www.gov.br/prf/pt-br/acesso-a-informacao/dados-abertos/dicionario-acidentes
- Apache Spark. **MLlib: Main Guide**. https://spark.apache.org/docs/latest/ml-guide.html

## Adiciona o Notebook no GitHub

In [ ]:
'''!apt-get install git -y -qq

!git config --global user.name "Arthursrosa-star"
!git config --global user.email "Arthur.s.rosa@iesb.edu.br"

!git clone https://github.com/arthursrosa-star/PDM_ML_SPARK_PRK.git
!cp "/content/PRF_Classificacao_Acidentes_SparkML_corrigido.ipynb" \
    "/content/PDM_ML_SPARK_PRK/"

%cd /content/PDM_ML_SPARK_PRK

!git add PRF_Classificacao_Acidentes_SparkML_corrigido.ipynb
!git commit -m "Adiciona notebook de classificação de acidentes da PRF com Spark ML"
!git push
'''